Run code from exploitation zone to prepare the csvs to be used by model

In [15]:
from exploitation_zone.prepare_data import generate_meta_trsa_csv, generate_height_trsa_csv

In [16]:
import pandas as pd
import os
import shutil


In [17]:
generate_meta_trsa_csv(
    "trusted_zone/preprocessed_data/08279_br_results_exploded.csv",
    "/Users/ceciliaperez/Documents/UPC-MD/TFM/Code/MARL-BuildingEnergyEstimation-main/data/data_config_1/meta_trsa2.csv"
)


Second Use Type Mapping:
    encoded            class_name
0         0          Agent office
1         1            Auditorium
2         2   Automobile commerce
3         3       Basic education
4         4      Butcher commerce
..      ...                   ...
75       75       Transformer hut
76       76      Undeveloped land
77       77  University education
78       78    Urbanization works
79       79             Warehouse

[80 rows x 2 columns]


,OBJECTID,YearBuilt1,HEIGHT,UseDescription,GrossFloorArea,orientation,street_width,second_type
0,1,1961,3,1,105.4,0.253521,0.064247,33
1,2,1941,3,1,154.52,0.253521,0.064380,33
2,3,1969,3,1,383.83,0.464789,0.000000,33
3,4,1968,3,1,176.82,0.760563,0.075879,33
4,5,1976,3,4,1714.82,0.887324,0.169876,15
...,...,...,...,...,...,...,...,...
37948,37949,2009,3,12,,0.816901,0.256318,48
37949,37950,1979,3,12,15.07,0.295775,0.000000,71
37950,37951,2009,3,2,6105.26,0.774648,0.079690,42
37951,37952,1967,6,5,4062.07,0.309859,0.679569,11


In [18]:
generate_height_trsa_csv(
    "trusted_zone/preprocessed_data/08279_br_results_exploded.csv",
    "/Users/ceciliaperez/Documents/UPC-MD/TFM/Code/MARL-BuildingEnergyEstimation-main/data/data_config_1/height_trsa.csv"
)

,OBJECTID,HEIGHT_norm
0,1,3
1,2,3
2,3,3
3,4,3
4,5,3
...,...,...
37948,37949,3
37949,37950,3
37950,37951,3
37951,37952,6


In [19]:

use_map = {
    1: "Residential",
    2: "Offices",
    3: "Industrial",
    4: "Cultural",
    5: "Commercial",
    6: "Entertainment_venues",
    7: "Healthcare_and_Charity",
    8: "Leisure_and_Hospitality",
    9: "Singular_building",
    10: "Sports_facilities",
    11: "Urbanization_land",
    12: "Warehouse_Parking"
}

In [21]:
# Paths
base_config = "../data/data_config_1"
source_folder = "../data/data_br/data_all"
residential_folder = os.path.join(base_config, "residential")
tertiary_folder = os.path.join(base_config, "tertiary")
tertiary_dest_base = "../data_br/data_all/data_tert_"

# Read data
df = pd.read_csv(os.path.join(base_config, "meta_trsa2.csv"))
df_h = pd.read_csv(os.path.join(base_config, "height_trsa.csv"))


In [22]:

for use_code, use_name in use_map.items():
    df_use = df[df['UseDescription'] == use_code]
    df_h_use = df_h[df_h['OBJECTID'].isin(df_use['OBJECTID'])]
    if use_code == 1:
        meta_dir = os.path.join(residential_folder, use_name)
    else:
        meta_dir = os.path.join(tertiary_folder, use_name)

    os.makedirs(meta_dir, exist_ok=True)
    df_use.to_csv(os.path.join(meta_dir, f"meta_trsa_{use_name}.csv"), index=False)
    df_h_use.to_csv(os.path.join(meta_dir, f"height_trsa_{use_name}.csv"), index=False)


In [23]:
import os
import shutil
import pandas as pd

source_folder = "../data/data_br/data_all_images/"

tertiary_csv_paths = {
    'residential': "../data/data_config_1/residential/Residential/meta_trsa_residential.csv",
    'offices': "../data/data_config_1/tertiary/Offices/meta_trsa_Offices.csv",
    'industrial': "../data/data_config_1/tertiary/Industrial/meta_trsa_Industrial.csv",
    'cultural': "../data/data_config_1/tertiary/Cultural/meta_trsa_Cultural.csv",
    'commercial': "../data/data_config_1/tertiary/Commercial/meta_trsa_Commercial.csv",
    'entertainment_venues': "../data/data_config_1/tertiary/Entertainment_venues/meta_trsa_Entertainment_venues.csv",
    'healthcare_and_charity': "../data/data_config_1/tertiary/Healthcare_and_Charity/meta_trsa_Healthcare_and_Charity.csv",
    'leisure_and_hospitality': "../data/data_config_1/tertiary/Leisure_and_Hospitality/meta_trsa_Leisure_and_Hospitality.csv",
    'singular_building': "../data/data_config_1/tertiary/Singular_building/meta_trsa_Singular_building.csv",
    'sports_facilities': "../data/data_config_1/tertiary/Sports_facilities/meta_trsa_Sports_facilities.csv",
    'urbanization_land': "../data/data_config_1/tertiary/Urbanization_land/meta_trsa_Urbanization_land.csv",        
    'warehouse_parking': "../data/data_config_1/tertiary/Warehouse_Parking/meta_trsa_Warehouse_Parking.csv"
}

# Load OBJECTIDs from each CSV into a dictionary of sets
tertiary_id_sets = {
    k: set(pd.read_csv(path)["OBJECTID"].astype(str).tolist())
    for k, path in tertiary_csv_paths.items()
}

final_folder = "../data/data_br/"

for filename in os.listdir(source_folder):
    if filename.endswith(".png"):
        object_id = os.path.splitext(filename)[0]

        for k, id_set in tertiary_id_sets.items():
            if object_id in id_set:
                # Keep category structure inside final_folder
                dest_dir = os.path.join(final_folder, k)
                os.makedirs(dest_dir, exist_ok=True)

                shutil.copy(
                    os.path.join(source_folder, filename),
                    os.path.join(dest_dir, filename)
                )
                break  # Stop after first match